In [ ]:
import pandas as pd
import re
import os
import yaml

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

### Import data

In [ ]:
# CNKI: Natural Science
CNKI_natural_JIF = pd.read_csv(dataset_config['path_processed'] + 'CNKI/01_CNKI_JIF_Natural_Science.csv', usecols=['刊名', '期刊综合类影响因子'])
CNKI_natural_JIF.rename(columns={'刊名': 'journal', '期刊综合类影响因子': 'JIF'}, inplace=True)

# CNKI: Social Science
CNKI_social_JIF = pd.read_csv(dataset_config['path_processed'] + 'CNKI/01_CNKI_JIF_Social_Science.csv', usecols=['刊名', '期刊综合类影响因子'])
CNKI_social_JIF.rename(columns={'刊名': 'journal', '期刊综合类影响因子': 'JIF'}, inplace=True)

# Concat
CNKI_JIF = pd.concat([CNKI_social_JIF, CNKI_natural_JIF], axis=0, ignore_index=True).drop_duplicates()
CNKI_JIF

In [ ]:
CNKI_raw_firms = pd.read_csv(dataset_config['path_processed'] + 'CNKI/02_CNKI_firm_paperid.csv', encoding='utf-8')
CNKI_raw_firms

### (1) Standard firm name

In [ ]:
# Define business suffixes and regional prefixes
business_suffixes = [
    '集团', '公司', '有限公司', '股份有限公司', '控股', '科技', '实业', '发展', 
    '有限责任公司', '制造', '工贸', '信息', '工程', '研究院', '研究所', '技术', '网络', '国际', '贸易'
]

region_prefixes = [
    '北京', '上海', '深圳', '广州', '天津', '重庆', '山东', '江苏', '浙江', '安徽', '福建', 
    '湖北', '湖南', '河北', '河南', '四川', '陕西', '山西', '江西', '广西', '黑龙江', '辽宁', 
    '吉林', '云南', '贵州', '甘肃', '海南', '新疆', '宁夏', '青海', '西藏', '内蒙古'
]

# Compile regular expressions
suffix_pattern = re.compile(r'(' + '|'.join(business_suffixes) + r')$', re.IGNORECASE)
region_pattern = re.compile(r'^(' + '|'.join(region_prefixes) + r')')

def clean_firm_name(name):
    if not name or not isinstance(name, str):
        return None  # Handle empty values or non-string data

    # Remove all special characters (excluding letters, numbers, and spaces)
    name = re.sub(r'[^\w\s]', '', name)

    # Remove regional prefixes
    name = region_pattern.sub('', name)

    # Remove business suffixes
    name = suffix_pattern.sub('', name)

    # Normalize to lowercase and trim leading/trailing spaces
    return name.strip().lower()

# Assuming CNKI_raw_firms is the dataframe
CNKI_raw_firms['firm_name_cleaned'] = CNKI_raw_firms['firm_name'].apply(clean_firm_name)

# Display processed data
CNKI_raw_firms


In [ ]:
# Generate unique firm_id based on firm_name_cleaned
CNKI_raw_firms['firm_id'] = pd.factorize(CNKI_raw_firms['firm_name'])[0]
CNKI_raw_firms

In [ ]:
# Specify the desired order of columns
desired_order = ['paperid', 'firm_id', 'year', 'entity_list', 'firm_uni_colla', 'funding', 'clc', 'journal', 'firm_name', 'firm_name_cleaned']
# Reorder columns
CNKI_firms = CNKI_raw_firms[desired_order]

CNKI_firms

### (2) Merge with JIF

In [ ]:
CNKI_firms_with_JIF = pd.merge(CNKI_firms, CNKI_JIF, on='journal', how='left')
CNKI_firms_with_JIF

### Export

In [ ]:
CNKI_firms_with_JIF.to_csv(dataset_config['path_processed'] + 'CNKI/03_CNKI_firm_paperid.csv', encoding='utf-8', index=False)